# Lab 11 — SOLUTION

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score

df = pd.read_csv("data/cleaned_hdb.csv")
X, y = df[["floor_area_sqm", "storey", "lease_commence"]], df["resale_price"]
Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.30, random_state=42)
Xval, Xte, yval, yte = train_test_split(Xtmp, ytmp, test_size=0.33, random_state=42)
assert len(set(Xtr.index) & set(Xte.index)) == 0

# buggy: fit scaler on FULL data before split
sc_bug = StandardScaler().fit(X)
Xtr_b, Xte_b = sc_bug.transform(Xtr), sc_bug.transform(Xte)
# correct: fit on train only
sc_ok = StandardScaler().fit(Xtr)
Xtr_o, Xte_o = sc_ok.transform(Xtr), sc_ok.transform(Xte)

knn = KNeighborsRegressor(n_neighbors=10)
knn.fit(Xtr_b, ytr); bug = r2_score(yte, knn.predict(Xte_b))
knn.fit(Xtr_o, ytr); ok = r2_score(yte, knn.predict(Xte_o))
print(f"leaky: {bug:.4f} | correct: {ok:.4f} | difference: {bug-ok:+.4f}")

**Teaching notes:** the score difference is usually small but real — the point is the *principle*, and the leakage grows with distribution shift. The disjoint-index assertion is the test the learners keep.